# Assignment 3 - K-Means Clustering

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc
from sklearn.cluster import k_means
from sklearn.preprocessing import StandardScaler
from pathlib import Path

results = Path('results/assignment3/kmeans')
results.mkdir(parents=True, exist_ok=True)

gene_counts = [10, 100, 1000, 5000, 10000]
k_values = [2, 3, 4, 5]

# Load Data

In [2]:
df = pd.read_csv('Data/SRP064298/SRP064298.tsv', sep='\t') # original dataset: change file to 'gene_data.tsv'
metadata = pd.read_csv('Data/SRP064298/metadata_SRP064298.tsv', sep='\t')
metadata = metadata.set_index('refinebio_accession_code')

# Match the metadata to the expression columns.
metadata = metadata.loc[df.columns[1:]]

# Keep only wild-type AL/RF samples, excluding KO samples.
wt_samples = metadata['refinebio_title'].str.startswith(('WT_AL_', 'WT_RF_'))
metadata = metadata.loc[wt_samples].copy()
metadata.index.name = 'sample_id'

# Extract AL or RF from titles such as WT_AL_ZT00_A_RNA.
groups = metadata['refinebio_title'].str.split('_').str[1].rename('group')
# Transpose to samples x genes, then keep the selected samples.
samples = df.drop(columns='Gene').T.loc[metadata.index]
samples.index.name = 'sample_id'

# Select variable genes

In [3]:
gene_variance = samples.var(axis=0)
gene_order = gene_variance[gene_variance > 0].sort_values(
    ascending=False, kind='mergesort'
).index

gene_ranking = pd.DataFrame({
    'gene_symbol': df['Gene'],
    'input_row': df.index,
    'variance': gene_variance,
}).loc[gene_order]
gene_ranking.index.name = 'gene_row'

for n_genes in gene_counts:
    gene_ranking.iloc[:n_genes].to_csv(results / f'top_{n_genes}_genes.csv')

gene_ranking.head(10)

,gene_symbol,input_row,variance
gene_row,,,
9917,ENSMUSG00000035202,9917,1604.011366
30934,ENSMUSG00000098973,30934,577.956662
29315,ENSMUSG00000096674,29315,562.459708
29326,ENSMUSG00000096688,29326,505.713260
7621,ENSMUSG00000030098,7621,491.777940
19896,ENSMUSG00000073867,19896,476.386073
29454,ENSMUSG00000096921,29454,443.752424
20001,ENSMUSG00000074115,20001,421.673108
9106,ENSMUSG00000032786,9106,387.049713


# Run K-Means Clustering

In [44]:
summaries = []
cluster_labels = pd.DataFrame(samples.index)

for n in gene_counts:
    selected_genes = gene_order[:n]
    scaled_data = StandardScaler().fit_transform(samples.loc[:, selected_genes])

    for i, k in enumerate(k_values):
        centroids, labels, inertia = k_means(X=scaled_data, n_clusters=k, random_state=42)
        run_name = f'kmeans_g{n}_k{k}'
        d = pd.DataFrame({'sample_id': samples.index, 'label': pd.Series(labels)})
        cluster_labels[run_name] = pd.Series(labels)
        dsg = d.value_counts('label')
        smallest_cluster = dsg.tail(1)
        largest_cluster = dsg.head(1)
        summaries.append({
            'run_name': run_name,
            'n_genes': n,
            'k': k,
            'n_runs': len(labels),
            'smallest_cluster': smallest_cluster.array[0],
            'largest_cluster': largest_cluster.array[0],
            'inertia_normal': inertia / n   # inertia score (sum of squared distances from nearest centroid) divided by # of genes
        })
    
summary = pd.DataFrame(summaries)
summary.to_csv(results / 'run_summary.csv', index=False)
cluster_labels.to_csv(results / 'cluster_labels.csv')

In [23]:
cluster_labels

,sample_id,kmeans_g10_k2,kmeans_g10_k3,kmeans_g10_k4,kmeans_g10_k5,kmeans_g100_k2,kmeans_g100_k3,kmeans_g100_k4,kmeans_g100_k5,kmeans_g1000_k2,...,kmeans_g1000_k4,kmeans_g1000_k5,kmeans_g5000_k2,kmeans_g5000_k3,kmeans_g5000_k4,kmeans_g5000_k5,kmeans_g10000_k2,kmeans_g10000_k3,kmeans_g10000_k4,kmeans_g10000_k5
0,SRR2534402,1,2,2,4,1,2,3,4,1,...,1,4,1,1,1,4,1,1,2,4
1,SRR2534403,1,2,2,4,1,2,3,4,1,...,1,4,1,1,1,4,1,1,2,4
2,SRR2534404,1,2,2,4,1,2,3,4,1,...,1,4,1,1,1,4,1,1,2,4
3,SRR2534405,1,2,2,4,1,2,3,4,1,...,1,4,1,1,1,4,1,1,2,4
4,SRR2534406,1,2,2,4,1,2,3,4,1,...,1,4,1,1,1,4,1,1,2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,SRR2535043,1,2,2,4,1,2,3,4,1,...,1,4,1,1,3,3,1,2,3,3
542,SRR2535044,1,2,2,4,1,2,3,4,1,...,1,4,1,1,3,3,1,2,3,3
543,SRR2535045,1,2,2,4,1,2,3,4,1,...,1,4,1,1,3,3,1,2,3,3
544,SRR2535046,1,2,2,4,1,2,3,4,1,...,1,4,1,1,3,3,1,2,3,3


# Perform a Chi-Squared test on each pair of clustering results

In [42]:
cl = cluster_labels.drop(columns=['sample_id'])
for i in range(len(cl.columns)-1):
    coli = cl.columns[i]
    if "k4" not in coli:
        continue
    for j in range(i+1, len(cl.columns)):
        colj = cl.columns[j]
        if "k4" not in colj:
            continue
        _, p_val, _, _ = sc.stats.chi2_contingency(pd.crosstab(cl[coli], cl[colj]))
        print(coli, colj, p_val)

kmeans_g10_k4 kmeans_g100_k4 6.668938200661079e-23
kmeans_g10_k4 kmeans_g1000_k4 2.1271966381710368e-33
kmeans_g10_k4 kmeans_g5000_k4 2.0170589853303937e-41
kmeans_g10_k4 kmeans_g10000_k4 1.394005506785993e-37
kmeans_g100_k4 kmeans_g1000_k4 5.073752837737049e-186
kmeans_g100_k4 kmeans_g5000_k4 8.87768994620371e-121
kmeans_g100_k4 kmeans_g10000_k4 3.924906465542903e-130
kmeans_g1000_k4 kmeans_g5000_k4 1.3363509014205735e-167
kmeans_g1000_k4 kmeans_g10000_k4 1.416628413654648e-119
kmeans_g5000_k4 kmeans_g10000_k4 1.4158284918786585e-190
